# 02 - TTC Bunching Modeling and Dashboard Outputs

## Scope
This notebook continues from the completed ingestion phase and builds the **next phase**:
- modeling-ready analytical table
- bunching target creation
- model-readiness EDA and visuals
- time-aware splitting and sklearn pipelines
- model comparison, tuning, evaluation
- dashboard-ready exports
- hotspot mapping (if coordinates available)


## 1) Project audit and assumptions
- Reuse existing raw/processed assets from the repository.
- Do **not** redo ingestion.
- If the available dataset is too small for model training, expand it deterministically for demonstration of the full workflow (no future leakage in engineered lag/rolling features).


In [2]:
from pathlib import Path
import sys
import pandas as pd

# Add repo root to path so src can be imported
sys.path.insert(0, str(Path('.').resolve().parent))

from src.metrics import (
    audit_project_state,
    build_modeling_table,
    create_target_summaries,
    time_aware_split,
    train_and_compare_models,
    tune_best_model,
    export_dashboard_tables,
    create_dashboard_plots,
    create_stop_hotspot_map,
    export_feature_importance,
)

audit = audit_project_state('.')
audit


{'raw_csv_files': [],
 'processed_csv_files': [],
 'notebooks': [],
 'has_raw_nvas_sample': False,
 'assumption': 'If no modeling-ready processed table exists, build one from existing raw/intermediate assets without redoing ingestion.'}

## 2) Build modeling-ready analytical table
Each row represents a route-stop-direction-time event with leakage-safe lag/rolling features.


In [5]:
import sys
print(sys.executable)

/home/codespace/.python/current/bin/python


In [6]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install gtfs-realtime-bindings protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gtfs-realtime-bindings]


In [7]:
from google.transit import gtfs_realtime_pb2
print("GTFS realtime import works")

GTFS realtime import works


In [9]:
from pathlib import Path
import subprocess
import requests
import pandas as pd
from google.transit import gtfs_realtime_pb2

PROJECT_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"],
    text=True
).strip())

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

raw_path = raw_dir / "ttc_nvas_vehicle_positions_sample.csv"

feed_url = "https://bustime.ttc.ca/gtfsrt/vehicles"
response = requests.get(feed_url, timeout=30)
response.raise_for_status()

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

rows = []

for entity in feed.entity:
    if entity.HasField("vehicle"):
        v = entity.vehicle
        rows.append({
            "entity_id": entity.id,
            "trip_id": v.trip.trip_id if v.HasField("trip") else None,
            "route_id": v.trip.route_id if v.HasField("trip") else None,
            "direction_id": v.trip.direction_id if v.HasField("trip") else None,
            "start_time": v.trip.start_time if v.HasField("trip") else None,
            "start_date": v.trip.start_date if v.HasField("trip") else None,
            "vehicle_id": v.vehicle.id if v.HasField("vehicle") else None,
            "vehicle_label": v.vehicle.label if v.HasField("vehicle") else None,
            "latitude": v.position.latitude if v.HasField("position") else None,
            "longitude": v.position.longitude if v.HasField("position") else None,
            "bearing": v.position.bearing if v.HasField("position") else None,
            "speed": v.position.speed if v.HasField("position") else None,
            "current_stop_sequence": getattr(v, "current_stop_sequence", None),
            "stop_id": getattr(v, "stop_id", None),
            "current_status": getattr(v, "current_status", None),
            "timestamp_utc": pd.to_datetime(
                v.timestamp, unit="s", errors="coerce", utc=True
            ) if getattr(v, "timestamp", None) else None,
            "occupancy_status": getattr(v, "occupancy_status", None),
        })

df_raw = pd.DataFrame(rows)
df_raw.to_csv(raw_path, index=False)

print("Saved to:", raw_path)
print("Shape:", df_raw.shape)
display(df_raw.head())
print(df_raw.columns.tolist())

Saved to: /workspaces/TTC-Bus-Reliability-and-Bunching-Monitor/data/raw/ttc_nvas_vehicle_positions_sample.csv
Shape: (1832, 17)


,entity_id,trip_id,route_id,direction_id,start_time,start_date,vehicle_id,vehicle_label,latitude,longitude,bearing,speed,current_stop_sequence,stop_id,current_status,timestamp_utc,occupancy_status
0,1,NaN,NaN,NaN,NaN,NaN,3640,,43.739265,-79.530197,349.0,14.30528,0,,2,2026-04-22 22:48:48+00:00,0
1,2,64715020,108,0.0,,,3638,,43.749866,-79.462318,145.0,4.02336,2,6160,2,2026-04-22 22:48:53+00:00,0
2,3,84969020,952,0.0,,,3630,,43.687363,-79.621109,163.0,0.00000,1,6900,0,2026-04-22 22:48:47+00:00,0
3,4,70702020,927,0.0,,,3632,,43.638393,-79.534767,39.0,0.00000,2,10064,2,2026-04-22 22:48:54+00:00,0
4,5,18609020,52,0.0,,,3635,,43.703461,-79.500153,73.0,0.00000,26,9659,0,2026-04-22 22:48:47+00:00,0


['entity_id', 'trip_id', 'route_id', 'direction_id', 'start_time', 'start_date', 'vehicle_id', 'vehicle_label', 'latitude', 'longitude', 'bearing', 'speed', 'current_stop_sequence', 'stop_id', 'current_status', 'timestamp_utc', 'occupancy_status']


In [10]:
df = build_modeling_table(
    raw_path=str(raw_path),
    output_path=str(PROJECT_ROOT / "data" / "processed" / "modeling_table.csv")
)

print("Modeling table shape:", df.shape)
display(df.head())

Modeling table shape: (1832, 37)


,entity_id,trip_id,route_id,direction_id,start_time,start_date,vehicle_id,vehicle_label,latitude,longitude,...,peak_flag,previous_headway,rolling_mean_headway,rolling_std_headway,headway_deviation,headway_ratio,route_instability_score,stop_instability_score,lag_bunching_1,bunching
0,866,15213020.0,7.0,0.0,NaN,NaN,9151,NaN,43.735790,-79.433609,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
1,480,65032020.0,7.0,0.0,NaN,NaN,9011,NaN,43.736767,-79.433861,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
2,489,136020.0,7.0,0.0,NaN,NaN,9022,NaN,43.755642,-79.438675,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
3,478,78498020.0,7.0,0.0,NaN,NaN,9017,NaN,43.715981,-79.428864,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
4,473,68033020.0,7.0,0.0,NaN,NaN,9012,NaN,43.790230,-79.446823,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0


## 3) Create bunching target and prevalence summaries
Target definition:
- bunching = 1 if observed_headway < 0.5 * scheduled_headway
- bunching = 0 otherwise


In [11]:
target_summaries = create_target_summaries(df, out_dir='data/processed')

print('Class balance:')
display(target_summaries['class_balance'])
print('Route prevalence sample:')
display(target_summaries['route_prevalence'].head())
print('Hour prevalence sample:')
display(target_summaries['hour_prevalence'].head())


Class balance:


,bunching,count,prevalence
0,0,1832,1.0


Route prevalence sample:


,route_id,bunching_rate
0,7.0,0.0
1,8.0,0.0
2,9.0,0.0
3,10.0,0.0
4,11.0,0.0


Hour prevalence sample:


,hour,bunching_rate
0,22,0.0


## 4) Model-readiness EDA checks


In [12]:
missing_summary = df.isna().sum().sort_values(ascending=False)
duplicate_count = int(df.duplicated().sum())
invalid_headway_rows = int((df['observed_headway'] <= 0).sum())
invalid_schedule_rows = int((df['scheduled_headway'] <= 0).sum())
timestamp_nulls = int(df['event_timestamp'].isna().sum())

print('Missing values summary:')
display(missing_summary.head(20))
print('Duplicate rows:', duplicate_count)
print('Impossible observed headways (<=0):', invalid_headway_rows)
print('Impossible scheduled headways (<=0):', invalid_schedule_rows)
print('Timestamp null rows:', timestamp_nulls)


Missing values summary:


start_time                 1832
headway_ratio              1832
observed_headway           1832
vehicle_label              1832
start_date                 1832
rolling_mean_headway       1832
rolling_std_headway        1832
headway_deviation          1832
previous_headway           1832
route_instability_score    1832
stop_instability_score     1832
stop_id                     360
route_id                    320
trip_id                     320
direction_id                320
entity_id                     0
current_stop_sequence         0
longitude                     0
bearing                       0
vehicle_id                    0
dtype: int64

Duplicate rows: 0
Impossible observed headways (<=0): 0
Impossible scheduled headways (<=0): 0
Timestamp null rows: 0


In [13]:
plot_paths = create_dashboard_plots(df, output_dir='data/processed/figures')
plot_paths


/workspaces/TTC-Bus-Reliability-and-Bunching-Monitor/src/metrics/modeling_pipeline.py:559: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(table["observed_headway"], label="Observed", fill=True)
/workspaces/TTC-Bus-Reliability-and-Bunching-Monitor/src/metrics/modeling_pipeline.py:560: UserWarning: Dataset has 0 variance; skipping density estimate. Pass `warn_singular=False` to disable this warning.
  sns.kdeplot(table["scheduled_headway"], label="Scheduled", fill=True)
/workspaces/TTC-Bus-Reliability-and-Bunching-Monitor/src/metrics/modeling_pipeline.py:563: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


['data/processed/figures/bunching_rate_by_route.png',
 'data/processed/figures/bunching_rate_by_hour.png',
 'data/processed/figures/bunching_rate_by_weekday.png',
 'data/processed/figures/top_15_worst_stops.png',
 'data/processed/figures/observed_vs_scheduled_distribution.png',
 'data/processed/figures/route_hour_heatmap.png',
 'data/processed/figures/numeric_correlation_heatmap.png']

### Quick interpretation guidance
- High route-level bunching bars indicate candidate reliability hotspots.
- Hourly peaks suggest operational pressure windows.
- Worst-stop chart can prioritize stop-level interventions.
- Route-hour heatmap helps target route-time combinations for dispatch actions.


## 5) Time-aware split (no random split)
Transit risk prediction must preserve chronology to avoid future leakage.


In [14]:
split = time_aware_split(df)
print('Train range:', split.train['event_timestamp'].min(), 'to', split.train['event_timestamp'].max(), 'rows=', len(split.train))
print('Validation range:', split.validation['event_timestamp'].min(), 'to', split.validation['event_timestamp'].max(), 'rows=', len(split.validation))
print('Test range:', split.test['event_timestamp'].min(), 'to', split.test['event_timestamp'].max(), 'rows=', len(split.test))


Train range: 2026-04-22 22:41:38+00:00 to 2026-04-22 22:48:52+00:00 rows= 1099
Validation range: 2026-04-22 22:48:52+00:00 to 2026-04-22 22:48:54+00:00 rows= 366
Test range: 2026-04-22 22:48:54+00:00 to 2026-04-22 22:48:56+00:00 rows= 367


## 6-8) sklearn pipeline, model comparison, and tuning
Models:
- Logistic Regression
- Random Forest
- HistGradientBoosting

Then tune a tree model with `RandomizedSearchCV` + `TimeSeriesSplit`.


In [16]:
# Check class balance before training
train_class_dist = split.train['bunching'].value_counts()
print("Training set class distribution:")
print(train_class_dist)

# Only train if both classes are present
if len(train_class_dist) < 2:
    print("WARNING: Training set has only one class. Skipping model training.")
    print("Recommendation: Collect more data or use stratified sampling.")
    model_metrics = None
    best_model = None
    best_info = None
else:
    model_metrics, best_model, best_info = train_and_compare_models(split, output_dir='data/processed')

if model_metrics is not None:
    model_metrics
model_metrics


Training set class distribution:
bunching
0    1099
Name: count, dtype: int64
Recommendation: Collect more data or use stratified sampling.


In [17]:
tuned_model, tuning_info = tune_best_model(split, output_dir='data/processed')
print('Best CV score:', tuning_info['best_cv_score'])
print('Best params:', tuning_info['best_params'])


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['observed_headway' 'headway_deviation' 'headway_ratio' 'previous_headway'
 'rolling_mean_headway' 'rolling_std_headway' 'route_instability_score'
 'stop_instability_score']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['observed_headway' 'headway_deviation' 'headway_ratio' 'previous_headway'
 'rolling_mean_headway' 'rolling_std_headway' 'route_instability_score'
 'stop_instability_score']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for 

Best CV score: nan
Best params: {'model__n_estimators': 350, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 10}


## 9-10) Overfitting/underfitting and evaluation
Use train/validation/test metrics (precision, recall, F1, ROC-AUC, PR-AUC) from `model_metrics.csv`.


In [19]:
route_stop_hour = export_dashboard_tables(df, output_dir='data/processed')
if tuned_model is not None and model_metrics is not None:
    feature_importance = export_feature_importance(
        tuned_model,
        feature_names=[
            'route_id','direction','stop_id','stop_name','day_of_week','hour','weekend_flag','peak_flag',
            'scheduled_headway','observed_headway','headway_deviation','headway_ratio','previous_headway',
            'rolling_mean_headway','rolling_std_headway','route_instability_score','stop_instability_score','lag_bunching_1'
        ],
        output_dir='data/processed'
    )
else:
    print("WARNING: Model was not trained with both classes. Skipping feature importance export.")
    feature_importance = pd.DataFrame()

display(feature_importance.head(20))


""


## 11) Findings template (fill after running all cells)
- **Best model:** 
- **Best hyperparameters:** 
- **Strongest predictors:** 
- **Highest-risk routes:** 
- **Highest-risk stops:** 
- **Highest-risk times:** 
- **Operational implication:**


## 12) Geographic hotspot map (if lat/lon available)


In [20]:
try:
    map_path = create_stop_hotspot_map(route_stop_hour['stop_summary'], output_path='data/processed/figures/stop_hotspots.html')
    print('Map saved:', map_path)
except Exception as exc:
    print('Map not generated:', exc)


Map saved: data/processed/figures/stop_hotspots.html


## 13-15) Deliverables checklist
After full run, expected artifacts:
- `data/processed/modeling_table.csv`
- `data/processed/route_summary.csv`
- `data/processed/stop_summary.csv`
- `data/processed/hour_summary.csv`
- `data/processed/model_metrics.csv`
- `data/processed/feature_importance.csv`
- `data/processed/figures/*.png`
- `data/processed/figures/stop_hotspots.html` (if coordinates available)
